# 創薬仮説 バッチ生成ノートブック
**Batch Drug Discovery Hypothesis Generator**

1つの疾患に対して複数の遺伝子を順番に処理し、レポートを自動生成します。

- **Step 1**: LLM・言語設定
- **Step 2**: 疾患を選択
- **Step 3**: 対象遺伝子リストを入力
- **Step 4**: バッチ実行（全遺伝子を自動処理）

## Step 1: LLM・言語設定

In [ ]:
import sys
sys.path.insert(0, '.')

for mod in list(sys.modules.keys()):
    if 'llm' in mod:
        del sys.modules[mod]

from llm.ollama_client import OllamaClient

MODEL = 'qwen2.5:14b'   # 例: 'llama3.1', 'qwen2.5:14b'
LANG  = 'en'            # 'ja' = 日本語 / 'en' = English

llm = OllamaClient(model=MODEL)
if llm.is_available():
    print(f'✓ Ollama ({MODEL}) 起動確認')
    print(f'  言語: {LANG}')
else:
    raise RuntimeError('⚠ Ollama に接続できません。ターミナルで ollama serve を実行してください。')

# ── コンテキスト情報量の設定 ──────────────────────────────
# 数値を変更するとLLMへ渡す情報量が変わります。
# 値を大きくすると仮説の根拠が豊富になりますが、生成時間も長くなります。

CONTEXT_CONFIG = dict(
    max_papers       = 5,    # 論文数（PubMed）
    abstract_chars   = 600,  # アブストラクト 1件あたりの文字数
    max_drugs        = 8,    # 薬剤数（ChEMBL + OpenTargets）
    max_gwas         = 5,    # GWAS ヒット数
    max_clinvar      = 5,    # ClinVar バリアント数
    max_interactions = 10,   # PPI インタラクター数（IntAct）
    max_trials       = 6,    # 臨床試験数（ClinicalTrials.gov）
    max_reactome     = 10,   # Reactome パスウェイ数
    gtex_top_n       = 5,    # GTEx 上位発現組織数
    hpa_top_n        = 8,    # Human Protein Atlas 組織数
    max_dgidb        = 8,    # DGIdb 薬剤-遺伝子相互作用数
    uniprot_chars    = 500,  # UniProt function 文字数
    uniprot_keywords = 10,   # UniProt キーワード数
    uniprot_go_terms = 8,    # UniProt GO term 数
)

# 軽量モード（バッチ速度優先）
# CONTEXT_CONFIG = dict(max_papers=2, abstract_chars=150, max_drugs=3,
#                       max_gwas=2, max_clinvar=2, max_interactions=6,
#                       max_trials=2, max_reactome=4, gtex_top_n=2,
#                       hpa_top_n=3, max_dgidb=3, uniprot_chars=150,
#                       uniprot_keywords=5, uniprot_go_terms=3)

from aggregator import DEFAULT_CONTEXT_CONFIG
print('\nコンテキスト設定:')
for k, v in CONTEXT_CONFIG.items():
    diff = f'  ← デフォルト: {DEFAULT_CONTEXT_CONFIG.get(k)}' if v != DEFAULT_CONTEXT_CONFIG.get(k) else ''
    print(f'  {k:<22} = {v}{diff}')

## Step 2: 疾患を選択

疾患名を入力して検索し、リストから選択してください。

In [ ]:
import requests
import ipywidgets as widgets
from IPython.display import display

OT_API = 'https://api.platform.opentargets.org/api/v4/graphql'

def _ot_search(keyword, entity):
    q = '''
    query ($q: String!, $e: [String!]) {
      search(queryString: $q, entityNames: $e, page: {index: 0, size: 15}) {
        hits { id name description entity }
      }
    }
    '''
    r = requests.post(OT_API, json={'query': q, 'variables': {'q': keyword, 'e': [entity]}}, timeout=15)
    r.raise_for_status()
    return [h for h in r.json()['data']['search']['hits'] if h['entity'] == entity]

selected_disease = {'id': None, 'name': None}

box  = widgets.Text(placeholder='疾患名を入力... (例: Duchenne muscular dystrophy)',
                    layout=widgets.Layout(width='440px'))
btn  = widgets.Button(description='検索', button_style='primary',
                      layout=widgets.Layout(width='70px'))
lst  = widgets.Select(options=[], rows=8,
                      layout=widgets.Layout(width='700px'))
stat = widgets.Label(value='疾患名を入力して「検索」を押してください')

def on_search(_):
    kw = box.value.strip()
    if not kw:
        stat.value = '⚠ キーワードを入力してください'; return
    stat.value = '検索中...'
    try:
        hits = _ot_search(kw, 'disease')
        if not hits:
            stat.value = f'「{kw}」に一致する疾患が見つかりませんでした'
            lst.options = []; return
        lst.options = [
            (f"{h['name']}  [{h['id']}]  {(h.get('description') or '')[:60]}", h)
            for h in hits
        ]
        stat.value = f'{len(hits)} 件見つかりました。リストから選択してください'
    except Exception as e:
        stat.value = f'エラー: {e}'

def on_select(change):
    val = change['new']
    if val:
        selected_disease['id']   = val['id']
        selected_disease['name'] = val['name']
        stat.value = f'✓ 選択済み: {val["name"]}  ({val["id"]})'

btn.on_click(on_search)
lst.observe(on_select, names='value')
display(widgets.VBox([widgets.HBox([box, btn]), lst, stat]))

## Step 3: 遺伝子リストを入力

`GENE_LIST_INPUT` を編集してセルを実行してください（1行1遺伝子 または カンマ区切り）。

In [ ]:
import re, requests
from IPython.display import display, HTML

# ── ここを編集してセルを実行 ──────────────────────────────
GENE_LIST_INPUT = """
ACACA
ACACB
"""
# ─────────────────────────────────────────────────────────

def _parse(text):
    return [g.strip().upper() for g in re.split(r'[,\n\r]+', text) if g.strip()]

def _validate_format(symbol):
    return bool(re.match(r'^[A-Z][A-Z0-9\-]{0,19}$', symbol))

def _check_hgnc(symbols):
    try:
        joined = '+OR+'.join(f'symbol:{s}' for s in symbols)
        r = requests.get(
            f'https://rest.genenames.org/search/{joined}',
            headers={'Accept': 'application/json'}, timeout=10,
        )
        found = {d['symbol'] for d in r.json().get('response', {}).get('docs', [])}
        return {s: s in found for s in symbols}
    except Exception:
        return {s: _validate_format(s) for s in symbols}

raw = _parse(GENE_LIST_INPUT)
if not raw:
    print('⚠ 遺伝子が入力されていません')
    GENE_LIST = []
else:
    fmt_ok       = [g for g in raw if _validate_format(g)]
    fmt_bad      = [g for g in raw if not _validate_format(g)]
    hgnc         = _check_hgnc(fmt_ok) if fmt_ok else {}
    confirmed    = [g for g in fmt_ok if hgnc.get(g, True)]
    unrecognized = [g for g in fmt_ok if not hgnc.get(g, True)]
    GENE_LIST    = confirmed + unrecognized

    rows = ''
    for g in confirmed:
        rows += f'<tr><td style="padding:3px 10px;color:#1a7f37">✓</td><td style="padding:3px 10px;font-weight:bold">{g}</td><td style="padding:3px 10px;color:#555;font-size:12px">HGNC confirmed</td></tr>'
    for g in unrecognized:
        rows += f'<tr><td style="padding:3px 10px;color:#9a6700">△</td><td style="padding:3px 10px;font-weight:bold">{g}</td><td style="padding:3px 10px;color:#9a6700;font-size:12px">未確認（処理は継続）</td></tr>'
    for g in fmt_bad:
        rows += f'<tr><td style="padding:3px 10px;color:#cf222e">✕</td><td style="padding:3px 10px;font-weight:bold">{g}</td><td style="padding:3px 10px;color:#cf222e;font-size:12px">形式エラー（スキップ）</td></tr>'

    note = f'<p style="color:#cf222e;font-size:12px;margin:4px 0 0">⚠ スキップ: {", ".join(fmt_bad)}</p>' if fmt_bad else ''
    display(HTML(f'''
    <b>✓ {len(GENE_LIST)} 遺伝子を登録</b>
    <table style="margin-top:6px;border-collapse:collapse;border:1px solid #ddd;font-size:13px">
      <thead><tr style="background:#f5f5f5">
        <th style="padding:3px 10px"></th>
        <th style="padding:3px 10px;text-align:left">遺伝子</th>
        <th style="padding:3px 10px;text-align:left">ステータス</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>{note}
    '''))

## Step 4: バッチ実行

全遺伝子を順番に処理します。各遺伝子のレポートは `reports/{GENE}_{DISEASE}/` に保存されます。

> ⚠ **注意**: LLM生成は遺伝子ごとに数分かかります。遺伝子数 × 生成時間を見込んでください。

### 各遺伝子に対して収集・使用するデータ

| データソース | 収集内容 | 設定キー |
|------------|---------|---------|
| PubMed | 論文・アブストラクト | `max_papers`, `abstract_chars` |
| OpenTargets | 遺伝子×疾患関連スコア | — |
| UniProt | タンパク質機能・局在・GO term | `uniprot_chars`, `uniprot_keywords`, `uniprot_go_terms` |
| GWAS Catalog | 遺伝的関連研究 | `max_gwas` |
| ClinVar | 病的変異 | `max_clinvar` |
| ChEMBL / OpenTargets | 既存薬・フェーズ・作用機序 | `max_drugs` |
| IntAct | タンパク質相互作用 (PPI) | `max_interactions` |
| gnomAD | 集団制約スコア pLI / LOEUF | — |
| GTEx | 組織別発現量 TPM | `gtex_top_n` |
| Human Protein Atlas | タンパク質発現・細胞内局在 | `hpa_top_n` |
| DGIdb | 薬剤–遺伝子相互作用 | `max_dgidb` |
| ClinicalTrials.gov | 臨床試験 | `max_trials` |
| AlphaFold DB | 構造信頼度 pLDDT | — |
| Reactome | 生物学的パスウェイ | `max_reactome` |
| PubChem / openFDA | 毒性アッセイ・副作用報告 | — |

In [ ]:
import sys, json, re
from datetime import datetime
from pathlib import Path
from IPython.display import display, Markdown, HTML

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ('collectors', 'aggregator', 'hypothesis', 'network')):
        del sys.modules[mod]

from aggregator import collect_all, build_llm_context
from hypothesis import generate_hypothesis, generate_presentation_eval
import network as net_mod

# ── 評価ヘルパー（4段階: Very High / High / Middle / Low） ──
def _to_mark(rating: str) -> str:
    if not rating or rating.strip() in ('', '...', '-', 'RATING'): return '—'
    s = rating.lower()
    if re.search(r'very.?high|非常に高', s): return 'VH'
    if re.search(r'\bhigh\b|^高$', s):       return 'H'
    if re.search(r'middle|moderate|中程度?', s): return 'M'
    if re.search(r'\blow\b|^低$|weak|弱い', s):  return 'L'
    if re.search(r'no.?data|データなし', s):     return '—'
    return '—'

MARK_COLOR = {'VH': '#0550ae', 'H': '#1a7f37', 'M': '#9a6700', 'L': '#cf222e', '—': '#aaa'}
MARK_BG    = {'VH': '#ddf4ff', 'H': '#e6ffed', 'M': '#fff8c5', 'L': '#ffebe9', '—': '#f6f8fa'}

EVAL_KEYS = [
    ('genetic_association',    'Genetic Association'),
    ('functional_association', 'Functional Association'),
    ('clinical_relevance',     'Clinical Relevance'),
    ('network_context',        'Expression / Network'),
    ('target_validity_overall','Overall'),
]

def _get_mark(ev: dict, key: str) -> tuple:
    item    = (ev or {}).get(key) or {}
    mark    = _to_mark(item.get('rating', ''))
    finding = (item.get('finding') or '').strip()
    bad     = {'...', 'FINDING', '仮説の一文要約', 'one sentence hypothesis summary',
               '一文', 'RATING', 'one-sentence overall summary', '総合評価を一文で'}
    if finding in bad: finding = ''
    return mark, finding

def _build_validity_table_md(ev: dict) -> str:
    lines = [
        '## Target Validity',
        '',
        '| Evaluation | Rating | Finding |',
        '|---|:---:|---|',
    ]
    for key, label in EVAL_KEYS:
        mark, finding = _get_mark(ev, key)
        bold = '**' if key == 'target_validity_overall' else ''
        lines.append(f'| {bold}{label}{bold} | {bold}{mark}{bold} | {finding} |')
    lines += [
        '',
        '<sub><sup>VH=Very High (clear association &amp; severity-linked) &nbsp;|&nbsp; '
        'H=High (clear association, pathway-direct) &nbsp;|&nbsp; '
        'M=Middle (explainable via PPI/pathway) &nbsp;|&nbsp; '
        'L=Low (no data)</sup></sub>',
        '',
    ]
    return '\n'.join(lines)

# ソース表示名の正規化
_SOURCE_LABEL = {
    'GO:BP': 'GO Biological Process',
    'GO:MF': 'GO Molecular Function',
    'GO:CC': 'GO Cellular Component',
    'KEGG':  'KEGG Pathway',
    'REAC':  'Reactome',
    'WP':    'WikiPathways',
    'TF':    'Transcription Factors',
    'MIRNA': 'miRNA targets',
    'HPA':   'Human Protein Atlas',
    'CORUM': 'Protein Complexes',
    'HP':    'Human Phenotype',
}

def _build_enrichment_md(enrichment: dict, top_per_source: int = 5) -> str:
    """エンリッチメント解析結果をソース別テーブルのMarkdownで返す。"""
    results = (enrichment or {}).get('results', [])
    if not results:
        return ''

    # ソース別にグループ化（p値昇順で上位 top_per_source 件）
    from collections import defaultdict
    by_source = defaultdict(list)
    for r in results:
        by_source[r['source']].append(r)

    # ソースの優先表示順
    SOURCE_ORDER = ['GO:BP', 'GO:MF', 'GO:CC', 'KEGG', 'REAC', 'WP', 'HP', 'TF', 'MIRNA', 'CORUM', 'HPA']
    ordered = [(s, by_source[s]) for s in SOURCE_ORDER if s in by_source]
    # 残りのソースを末尾に追加
    for s in sorted(by_source):
        if s not in SOURCE_ORDER:
            ordered.append((s, by_source[s]))

    lines = ['## Functional Enrichment (g:Profiler, FDR < 0.05)', '']
    for source, terms in ordered:
        label = _SOURCE_LABEL.get(source, source)
        lines.append(f'### {label}')
        lines.append('')
        lines.append('| Term | p-value | Genes (overlap) |')
        lines.append('|---|:---:|---|')
        for t in terms[:top_per_source]:
            p   = f"{t['p_value']:.2e}"
            n   = t.get('intersection_size', 0)
            gs  = ', '.join(t.get('genes', [])[:8])
            tid = t.get('term_id', '')
            name = t['term_name'][:70]
            lines.append(f'| {name} `{tid}` | {p} | {gs} ({n}) |')
        lines.append('')

    total_sig = len(results)
    lines.append(f'<sub><sup>有意項目合計: {total_sig} 件 (FDR&lt;0.05) — '
                 f'[g:Profiler](https://biit.cs.ut.ee/gprofiler/gost)</sup></sub>')
    lines.append('')
    return '\n'.join(lines)

# ── 入力チェック ──────────────────────────────────────────
if not selected_disease.get('name'):
    raise ValueError('⚠ Step 2 で疾患を選択してください')
if not GENE_LIST:
    raise ValueError('⚠ Step 3 で遺伝子を入力してセルを実行してください')

DISEASE    = selected_disease['name']
DISEASE_ID = selected_disease['id']

print(f'疾患: {DISEASE}  ({DISEASE_ID})')
print(f'対象遺伝子 ({len(GENE_LIST)}件): {", ".join(GENE_LIST)}')
print('=' * 60)

results_summary = []

for idx, GENE in enumerate(GENE_LIST, 1):
    print(f'\n[{idx}/{len(GENE_LIST)}] {GENE} × {DISEASE}')
    print('-' * 50)

    try:
        raw_evidence = collect_all(GENE, DISEASE, verbose=True, disease_id=DISEASE_ID)
    except Exception as e:
        print(f'  ✗ データ収集失敗: {e}')
        results_summary.append({'gene': GENE, 'status': f'データ収集失敗: {e}', 'eval': {}})
        continue

    print('  PPIネットワーク構築中...')
    try:
        ppi_graph = net_mod.build_ppi_network(GENE, use_biogrid=False)
        network_enrichment = net_mod.run_network_enrichment(ppi_graph) if ppi_graph else {}
        if ppi_graph:
            print(f'  ✓ PPI: {ppi_graph.number_of_nodes()} nodes, {ppi_graph.number_of_edges()} edges')
        n_enr = len((network_enrichment or {}).get('results', []))
        if n_enr:
            print(f'  ✓ エンリッチメント: {n_enr} 有意項目')
    except Exception:
        ppi_graph = None
        network_enrichment = {}

    context = build_llm_context(raw_evidence, config=CONTEXT_CONFIG)
    if ppi_graph:
        context += '\n\n' + net_mod.network_summary_for_llm(
            ppi_graph, GENE, network_enrichment, max_partners=8, max_terms=10)
    print(f'  コンテキスト: {len(context):,} 文字')

    print('  仮説生成中... (ストリーミング)\n')
    try:
        def _cb(token): print(token, end='', flush=True)
        hypothesis = generate_hypothesis(GENE, DISEASE, context, llm, lang=LANG, stream_callback=_cb)
        print(f'\n  ✓ 仮説生成完了 ({len(hypothesis):,} 文字)')
    except Exception as e:
        print(f'\n  ✗ 仮説生成失敗: {e}')
        results_summary.append({'gene': GENE, 'status': f'仮説生成失敗: {e}', 'eval': {}})
        continue

    print('  評価カード生成中...（本文の結論に合わせて評価）')
    eval_result = {}
    try:
        eval_result = generate_presentation_eval(GENE, DISEASE, context, llm, lang=LANG, hypothesis=hypothesis) or {}
        if eval_result:
            for k, lbl in EVAL_KEYS:
                print(f'    {lbl}: {(eval_result.get(k) or {}).get("rating","—")}')
        else:
            print('  ⚠ 評価カード: JSONパース失敗')
    except Exception as e:
        print(f'  ⚠ 評価カード生成エラー: {e}')

    # ── 個別レポート保存 ──────────────────────────────────
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    pair_dir  = Path('reports') / f'{GENE}_{DISEASE.replace(" ", "_")}'
    pair_dir.mkdir(parents=True, exist_ok=True)

    lang_suffix = 'JA' if LANG == 'ja' else 'EN'
    rpt_path    = pair_dir / f'{timestamp}_{lang_suffix}.md'

    # PPIネットワーク画像
    ppi_md = ''
    if ppi_graph and ppi_graph.number_of_edges() > 0:
        try:
            img_name = f'{timestamp}_ppi.png'
            saved = net_mod.render_ppi_image(
                ppi_graph, GENE, str(pair_dir / img_name),
                enrichment=network_enrichment, max_nodes=24)
            if saved:
                ppi_md = (f'## PPI Network\n\n'
                          f'![PPI network of {GENE}]({img_name})\n\n'
                          f'<sub><sup>★ = {GENE} (target, 上部) ／ 下部 = PPI パートナー'
                          f'（色 = enrichment 上位パスウェイ）</sup></sub>\n\n')
                print(f'  ✓ PPI画像: {pair_dir / img_name}')
        except Exception as e:
            print(f'  ⚠ PPI画像生成エラー: {e}')

    # エンリッチメント解析結果
    enrichment_md = _build_enrichment_md(network_enrichment)

    with open(rpt_path, 'w', encoding='utf-8') as f:
        f.write(f'# Drug Discovery Hypothesis: {GENE} × {DISEASE}\n')
        f.write(f'Generated: {datetime.now().isoformat()}  |  Language: {LANG}\n\n---\n\n')
        f.write(_build_validity_table_md(eval_result))
        f.write('---\n\n')
        if ppi_md:
            f.write(ppi_md)
        if enrichment_md:
            f.write(enrichment_md)
            f.write('---\n\n')
        f.write(hypothesis)
        f.write('\n\n---\n\n## Evidence Context\n\n')
        f.write(context)

    if eval_result:
        with open(pair_dir / f'{timestamp}_eval.json', 'w', encoding='utf-8') as f:
            json.dump(eval_result, f, ensure_ascii=False, indent=2)
    with open(pair_dir / f'{timestamp}_raw.json', 'w', encoding='utf-8') as f:
        json.dump(raw_evidence, f, ensure_ascii=False, indent=2, default=str)

    print(f'  ✓ 保存: {rpt_path}')
    results_summary.append({
        'gene': GENE, 'status': '✓ 完了',
        'path': str(rpt_path), 'eval': eval_result,
    })

# ── バッチサマリーテーブル（行=評価項目、列=遺伝子） ─────
print('\n' + '=' * 60)
print('バッチ完了 — サマリー生成中...')

legend_html = '''
<div style="display:flex;gap:14px;flex-wrap:wrap;font-size:12px;color:#444;margin:0 0 10px">
  <span><b style="background:#ddf4ff;color:#0550ae;padding:2px 7px;border-radius:3px">VH</b>
    Very High — 関連明確かつ悪性度・進行と関連</span>
  <span><b style="background:#e6ffed;color:#1a7f37;padding:2px 7px;border-radius:3px">H</b>
    High — 関連明確・パスウェイ直結</span>
  <span><b style="background:#fff8c5;color:#9a6700;padding:2px 7px;border-radius:3px">M</b>
    Middle — PPI/パスウェイで説明可能</span>
  <span><b style="background:#ffebe9;color:#cf222e;padding:2px 7px;border-radius:3px">L</b>
    Low — 情報なし</span>
</div>
'''
gene_th = ''.join(
    f'<th style="padding:6px 16px;text-align:center;border:1px solid #ddd;background:#f5f5f5">'
    f'{r["gene"]}</th>'
    for r in results_summary
)
body_rows = ''
for key, label in EVAL_KEYS:
    is_overall  = (key == 'target_validity_overall')
    row_bg      = 'background:#ececec;' if is_overall else ''
    label_style = 'font-weight:bold;' if is_overall else ''
    cells = ''
    for r in results_summary:
        mark, tip = _get_mark(r.get('eval', {}), key)
        bg    = MARK_BG.get(mark, '#f6f8fa')
        color = MARK_COLOR.get(mark, '#aaa')
        title = f' title="{tip}"' if tip else ''
        cells += (
            f'<td style="padding:7px 16px;text-align:center;border:1px solid #ddd;'
            f'background:{bg};font-weight:bold;color:{color};font-size:15px"{title}>'
            f'{mark}</td>'
        )
    body_rows += (
        f'<tr style="{row_bg}">'
        f'<td style="padding:7px 12px;border:1px solid #ddd;white-space:nowrap;{label_style}">'
        f'{label}</td>{cells}</tr>'
    )

display(HTML(f'''
<h3 style="margin-bottom:6px">Target Validity Summary — {DISEASE}</h3>
{legend_html}
<table style="border-collapse:collapse;font-size:13px">
  <thead><tr>
    <th style="padding:6px 12px;text-align:left;border:1px solid #ddd;background:#f5f5f5">Evaluation</th>
    {gene_th}
  </tr></thead>
  <tbody>{body_rows}</tbody>
</table>
<p style="font-size:11px;color:#888;margin:6px 0 0">※ セルにカーソルを合わせると根拠が表示されます</p>
'''))

now       = datetime.now().strftime('%Y-%m-%d %H:%M')
gene_cols = [r['gene'] for r in results_summary]
md_lines  = [
    f'# Target Validity Summary — {DISEASE}',
    f'Generated: {now}', '',
    'VH=Very High  H=High  M=Middle  L=Low  —=No data', '',
    '| Evaluation | ' + ' | '.join(gene_cols) + ' |',
    '|------------|' + '|'.join([':----------:'] * len(results_summary)) + '|',
]
for key, label in EVAL_KEYS:
    marks = [_get_mark(r.get('eval', {}), key)[0] for r in results_summary]
    md_lines.append(f'| **{label}** | ' + ' | '.join(marks) + ' |')

summary_md      = '\n'.join(md_lines)
ts              = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_md_path = Path('reports') / f'{DISEASE.replace(" ","_")}_summary_{ts}.md'
summary_md_path.write_text(summary_md, encoding='utf-8')
print(f'サマリーMD保存: {summary_md_path}')
display(Markdown(summary_md))